In [ ]:
import random
from datasets import load_dataset
import os

# ===========================================================================
# ✨ 코딩 튜터님의 오늘의 과제: AI 안전 필터링 시뮬레이션 ✨
#
# 📝 데이터셋 이름: HarethahMo/extended-refusal
# 🌍 데이터셋 주제: 모델 거절 (Refusal) / AI 안전성 (Safety Alignment)
# 💡 데이터 의미: 이 데이터셋은 거부해야 할 요청(Instruction + Input)에 대해
#               LLM이 안전하고 적절하게 응답하는 방법(Output)을 학습시키는 데 사용됩니다.
#               쉽게 말해, AI가 위험하거나 부적절한 질문에 "죄송하지만 이 요청은 처리할 수 없습니다"
#               라고 당당하게 말하도록 가르치는 자료입니다!
#
# 🚀 실습 목표: 주어진 데이터를 탐색하며, AI가 어떻게 안전하게 답변을 생성하는지
#              '안전 가드(Safety Guard)' 시뮬레이션을 통해 눈으로 익혀봅시다!
# ===========================================================================

# --- 설정 값 ---
DATASET_NAME = "HarethahMo/extended-refusal"
SAMPLE_COUNT = 10  # 전체 데이터를 돌리기에는 많으니, 재미로 10개만 살펴볼 거예요!

print("🤖 안녕하세요, 코딩 튜터 AI입니다! 준비되셨나요? 😉")
print("오늘의 미션은 AI의 '상황 판단력'을 키워보는 거예요. 데이터셋을 로드하며 같이 탐험해 봅시다!")
print("-" * 60)

# ---------------------------------------------------------------------------
# 🛠️ 1단계: 데이터셋 로드 (스트리밍 방식을 우선 시도!)
# ---------------------------------------------------------------------------
dataset = None
try:
    # 스트리밍(streaming=True)은 데이터셋을 메모리에 다 올리지 않고, 필요한 만큼만 가져와서
    # 매우 큰 데이터셋을 효율적으로 다룰 수 있게 해줍니다. (AI 개발의 핵심 기술 중 하나!)
    print("💾 데이터셋을 스트리밍 모드로 로드 시도 중... (속도 측정 중!)")
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 성공! 스트리밍 모드로 데이터셋을 가져왔습니다. (아주 빠르죠?)")

except Exception as e:
    # 만약 스트리밍 로딩이 어떤 이유로 실패한다면 (네트워크 문제 등),
    # 일반 모드로 로딩을 시도하여 작업을 중단시키지 않도록 방지합니다.
    print(f"⚠️ 스트리밍 로드 실패 ({e}). 일반 다운로드 모드로 전환합니다.")
    try:
        # 학습 용도인 'train' 스플릿 전체를 다운로드하여 메모리에 올립니다.
        dataset = load_dataset(DATASET_NAME, split='train')
        print("✅ 성공! 일반 다운로드 방식으로 데이터셋을 준비했습니다. (조금 느릴 수 있어요)")
    except Exception as e_fallback:
        print(f"💥 죄송합니다, 데이터셋 로드에 실패했습니다: {e_fallback}")
        exit()

# ---------------------------------------------------------------------------
# 🚀 2단계: 샘플 데이터 확보 및 반복 준비
# ---------------------------------------------------------------------------

# 🚨 중요 규칙: len()을 사용하지 않고, take()와 iter() 패턴을 사용합니다!
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)입니다.
    print(f"\n🧭 최대 {SAMPLE_COUNT}개의 샘플을 가져와서 탐험을 시작합니다!")
    sampled_dataset_iterator = iter(dataset.take(SAMPLE_COUNT))
else:
    # 일반 데이터셋 (Dataset)인 경우입니다.
    print(f"\n🧭 최대 {SAMPLE_COUNT}개의 샘플을 가져와서 탐험을 시작합니다!")
    # list(dataset.take(N))를 통해 리스트로 확보합니다.
    sample_data_list = list(dataset.take(SAMPLE_COUNT))
    sampled_dataset_iterator = iter(sample_data_list)

# ---------------------------------------------------------------------------
# 💡 3단계: '안전 가드' 시뮬레이션 실습
# ---------------------------------------------------------------------------

print("\n" + "=" * 60)
print("🛡️ [미션 수행] AI 안전 가드 시뮬레이션 (Safety Guard Simulation)")
print("=================================================================\n")

sample_count = 0
# 반복자가 살아있는 동안 루프를 실행합니다.
sample_data = next(sampled_dataset_iterator, None)

while sample_data is not None and sample_count < SAMPLE_COUNT:
    sample_count += 1
    
    # 데이터를 구조적으로 이해하기 위해 변수에 저장합니다.
    instruction = sample_data['instruction']
    input_query = sample_data['input']
    refusal_output = sample_data['output']
    
    # 📝 코드 해석 (튜터의 친절한 해설):
    # 'instruction'과 'input'이 바로 사용자가 AI에게 던진 질문/요청(The Prompt)입니다.
    # 'refusal_output'은 이 요청을 받고 AI가 "놉!"이라고 대답하는 안전한 답변입니다.
    
    print(f"\n==================> 샘플 {sample_count}번째 분석 <=================")
    
    # 🔎 1. 요청 분석 (탐색 능력 강화)
    print("➡️ 🕵️‍♂️ 사용자 요청 (The Prompt):")
    print(f"  [지시사항]: {instruction[:40]}...")
    print(f"  [사용자 입력]: {input_query[:60]}...")
    
    # 🔍 2. 핵심 질문 던지기 (데이터 패턴 분석)
    print("\n🤔 이 데이터의 핵심은 무엇일까요?")
    print("   -> 사용자의 요청(Prompt)이 부적절하거나 민감할 때, AI가 어떻게 안전한 응답을 하는지 패턴을 학습하는 것입니다!")
    
    # 🌈 3. 안전 가드 작동! (실제 실습 부분)
    print("\n✨ AI 안전 가드 작동! (Refusal Output)")
    print("-------------------------------------------")
    print(f"🤖 AI가 출력한 안전한 답변: {refusal_output[:80]}...")
    
    # 💡 4. 추가 분석 (Label 필드 사용)
    # 'label' 필드는 이 샘플이 어떤 유형의 위반에 속하는지 알려주는 '태그' 역할을 할 수 있습니다.
    label = sample_data.get('label', 'N/A')
    print(f"\n[✨ 분석 결과]: 이 요청은 '{label}' 유형의 문제를 다루고 있습니다.")
    print("-----------------------------------------------------------------")

    # 다음 샘플로 이동
    sample_data = next(sampled_dataset_iterator, None)

print("\n🎉🎉🎉 축하합니다! 미션 성공! 🎉🎉🎉")
print("✨ 오늘 데이터를 탐험해보니, AI가 단순히 답변만 하는 것이 아니라,")
print("   '윤리적인 필터'를 통해 위험을 회피하고 안전한 대화를 이끌어내는 것이 핵심임을 알게 되었죠? 👍")
print("   이것이 바로 LLM Alignment의 매력이랍니다!")